# CS 3110/5990: Data Privacy
## Homework 4

In [126]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')

KeyboardInterrupt: 

In [156]:
BASE_CLIP = 100 * 1000

## Question 1 (10 points)

Complete the definition of `dp_sum_capgain` below. Your definition should compute a differentially private sum of the "Capital Gain" column of the `adult` dataset, and have a total privacy cost of `epsilon`.

In [158]:
def dp_sum_capgain(epsilon):
    clipped = adult['Capital Gain'].clip(upper=BASE_CLIP)
    return laplace_mech(clipped.sum(), sensitivity=BASE_CLIP, epsilon=epsilon)

dp_sum_capgain(1.0)

np.float64(35092796.990063794)

In [159]:
# TEST CASE for question 1

real_sum = adult['Capital Gain'].sum()
r1 = np.mean([pct_error(real_sum, dp_sum_capgain(0.1)) for _ in range(100)])
r2 = np.mean([pct_error(real_sum, dp_sum_capgain(1.0)) for _ in range(100)])
r3 = np.mean([pct_error(real_sum, dp_sum_capgain(10.0)) for _ in range(100)])

print("Average errors:", r1, r2, r3)
print()
assert r1 > 0
assert r2 > 0
assert r3 > 0
assert r1 < 10
assert r2 < 2
assert r3 < 0.2

Average errors: 2.863941064097467 0.2881285309630241 0.029104264389763088



## Question 2 (10 points)

In 2-5 sentences each, answer the following:

- What clipping parameter did you use in your definition of `dp_sum_capital`, and why?

I used a clipping parameter of 150,000. It removed enough values to keep sensitivity reasonable while not being so large that it meaningfully changes the actual sum.

- What was the sensitivity of the query you used in `dp_sum_capital`, and how is it bounded?

The sensitivity was 150,000. It is bounded by clipping each person to at most 150,000

- Argue that your definition of `dp_sum_capital` has a total privacy cost of `epsilon`

The laplace mechanisim was applied once with a parameter of epsilon. 1 [applied 1 time] * 1 [epsilon of 1] = 1 [total privacy cost of 1]

YOUR ANSWER HERE

## Question 3 (10 points)

Complete the definition of `dp_avg_capgain` below. Your definition should compute a differentially private average (mean) of the "Capital Gain" column of the adult dataset, and have a **total privacy cost of epsilon**.

In [131]:
def dp_avg_capgain(epsilon):
    pSum = dp_sum_capgain(epsilon/2)

    clipped = adult['Capital Gain'].clip(upper=BASE_CLIP)
    pCount = laplace_mech(len(clipped), sensitivity=1, epsilon=epsilon/2)

    avg = pSum / pCount

    return avg


dp_avg_capgain(1.0)

np.float64(1076.8883659791459)

In [132]:
# TEST CASE for question 3

real_avg = adult['Capital Gain'].mean()
r1 = np.mean([pct_error(real_avg, dp_avg_capgain(0.1)) for _ in range(100)])
r2 = np.mean([pct_error(real_avg, dp_avg_capgain(1.0)) for _ in range(100)])
r3 = np.mean([pct_error(real_avg, dp_avg_capgain(10.0)) for _ in range(100)])

print("Average errors:", r1, r2, r3)

assert r1 > 0
assert r2 > 0
assert r3 > 0
assert r1 < 20
assert r2 < 4
assert r3 < 0.4

Average errors: 5.198239851418219 0.5893161048171935 0.059543348460570626


In [155]:
def dp_avg_capgain_setClip(epsilon, clip):
    pSum = dp_sum_capgain(epsilon/2)

    clipped = adult['Capital Gain'].clip(upper=clip)
    pCount = laplace_mech(len(clipped), sensitivity=1, epsilon=epsilon/2)

    avg = pSum / pCount

    return avg

clips = [0, 50000, 75000, 100000]
for clip in clips:
    r1 = np.mean([pct_error(real_avg, dp_avg_capgain_setClip(0.1, clip)) for _ in range(100)])
    r2 = np.mean([pct_error(real_avg, dp_avg_capgain_setClip(1, clip)) for _ in range(100)])
    r3 = np.mean([pct_error(real_avg, dp_avg_capgain_setClip(10, clip)) for _ in range(100)])

    print(f"Average errors, cliping at ({clip}): ep = 0.1 {r1}, ep = 1 {r2}, ep = 10 {r3}")

print("")
print(f"All Adults: {len(adult)}")
for clip in clips:
    print(f"Capital Gain > {clip}: {len(adult[adult['Capital Gain'] > clip])}")

Average errors, cliping at (0): ep = 0.1 5.830161247597123, ep = 1 0.6461884028791627, ep = 10 0.06340698603640348
Average errors, cliping at (50000): ep = 0.1 6.500274885156733, ep = 1 0.5773104733015665, ep = 10 0.05509080642021652
Average errors, cliping at (75000): ep = 0.1 6.056248545298124, ep = 1 0.5905573501450978, ep = 10 0.05836497969703409
Average errors, cliping at (100000): ep = 0.1 5.756234915912366, ep = 1 0.520380469067532, ep = 10 0.054917329471107675

All Adults: 32561
Capital Gain > 0: 2712
Capital Gain > 50000: 159
Capital Gain > 75000: 159
Capital Gain > 100000: 0


## Question 4 (10 points)

In 2-5 sentences each, answer the following:

- Argue that your definition of `dp_avg_capgain` has a total privacy cost of `epsilon`

The laplace mechanisim was applied twice with a parameter of epsilon divided by 2. 2 [applied 2 times] * 0.5 [epsilon of 1/2] = 1 [total privacy cost of 1]

- For sums and averages, which seems to be more important for accuracy - the value of the clipping parameter $b$ or the scale of the noise added? Why?

Here the scale of the noise seems to make impact accuracy more than the value of the clipping parameter. 
- Do you think the answer to the previous point will be true for every dataset? Why or why not?

YOUR ANSWER HERE

## Question 5 (20 points)

Write a function `auto_avg` that returns the differentially private average of a Pandas series `s`. Your function should automatically determine the clipping parameter `b`, and should enforce differential privacy for a **total privacy cost** of `epsilon`. You can assume that all values are non-negative (i.e. 0 or greater).

In [98]:
# YOUR CODE HERE
raise NotImplementedError()

NotImplementedError: 

In [ ]:
# TEST CASE for question 5
r1 = np.mean([pct_error(adult['Age'].mean(), auto_avg(adult['Age'], 1.0)) for _ in range(20)])
r2 = np.mean([pct_error(adult['Capital Gain'].mean(), auto_avg(adult['Capital Gain'], 1.0)) for _ in range(20)])
r3 = np.mean([pct_error(adult['fnlwgt'].mean(), auto_avg(adult['fnlwgt'], 1.0)) for _ in range(20)])

print('Average errors:', r1, r2, r3)
assert r1 > 0
assert r2 > 0
assert r3 > 0
assert r1 < 1
assert r2 < 100
assert r3 < 1

## Question 6

In 2-5 sentences each, answer the following:

- Explain your strategy for implementing `auto_avg`
- Argue informally that your definition of `auto_avg` has a total privacy cost of `epsilon`
- Did your solution work well for all three example columns? If it did not work well on any of them, why not?
- When is your solution likely to *not* work well? (i.e. what properties does the data have to have, in order for your solution to not work well?)

YOUR ANSWER HERE